# OmniParser v2 — Google Colab Demo

> **GPU 런타임 필수**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 저장

## 0. GPU 런타임 확인

In [1]:
import subprocess, sys

# nvidia-smi 로 GPU 확인
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('❌ GPU를 찾을 수 없습니다.')
    print('   상단 메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 다시 실행하세요.')
    sys.exit(1)
else:
    print(result.stdout)

import torch
if not torch.cuda.is_available():
    print('❌ torch.cuda.is_available() = False')
    print('   GPU 런타임이 활성화되어 있지 않습니다.')
    sys.exit(1)

print(f'✅ GPU 확인 완료')
print(f'   Device : {torch.cuda.get_device_name(0)}')
print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'   CUDA   : {torch.version.cuda}')

Sat May 30 16:43:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 레포 클론 및 패키지 설치

약 5~10분 소요됩니다.

In [ ]:
import os

if not os.path.exists('/content/OmniParser'):
    !git clone https://github.com/microsoft/OmniParser /content/OmniParser
else:
    print('이미 클론되어 있습니다.')

os.chdir('/content/OmniParser')
print('작업 디렉토리:', os.getcwd())

In [ ]:
# paddleocr 3.x 호환 패치: PaddleOCR() 인자를 lang='en' 하나만 남김
import re

utils_path = '/content/OmniParser/util/utils.py'
with open(utils_path, 'r') as f:
    src = f.read()

# paddle_ocr = PaddleOCR(...) 블록을 통째로 교체
patched, n = re.subn(
    r'paddle_ocr\s*=\s*PaddleOCR\([^)]*\)',
    "paddle_ocr = PaddleOCR(lang='en')",
    src,
    flags=re.DOTALL
)

if n:
    with open(utils_path, 'w') as f:
        f.write(patched)
    print(f'✅ PaddleOCR 패치 완료 ({n}곳)')
else:
    print('ℹ️  패치 대상 없음')

In [ ]:
# Colab에는 torch/torchvision/numpy가 이미 설치되어 있으므로 제외
# paddlepaddle은 공식 PyPI CPU 버전 사용 (Tsinghua 미러 속도 문제 회피)
!pip install -q \
    easyocr \
    transformers \
    ultralytics==8.3.70 \
    supervision==0.18.0 \
    timm \
    einops==0.8.0 \
    accelerate \
    paddlepaddle \
    paddleocr

## 2. 모델 가중치 다운로드

HuggingFace에서 OmniParser-v2.0 체크포인트를 다운로드합니다. (~1.5 GB)

In [ ]:
!pip install -q -U huggingface_hub

import os
os.makedirs('weights', exist_ok=True)

files = [
    'icon_detect/train_args.yaml',
    'icon_detect/model.pt',
    'icon_detect/model.yaml',
    'icon_caption/config.json',
    'icon_caption/generation_config.json',
    'icon_caption/model.safetensors',
]

for f in files:
    !hf download microsoft/OmniParser-v2.0 "{f}" --local-dir weights

# florence2 경로로 rename
if os.path.exists('weights/icon_caption') and not os.path.exists('weights/icon_caption_florence'):
    os.rename('weights/icon_caption', 'weights/icon_caption_florence')

print('\n가중치 파일 확인:')
!find weights -type f | sort

## 3. 모델 로드

In [ ]:
import sys
sys.path.insert(0, '/content/OmniParser')

import torch
from PIL import Image
from util.utils import get_som_labeled_img, check_ocr_box, get_caption_model_processor, get_yolo_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {device}')

# YOLO 아이콘 감지 모델
som_model = get_yolo_model('weights/icon_detect/model.pt')
som_model.to(device)
print('✅ YOLO 모델 로드 완료')

# Florence2 캡션 모델
caption_model_processor = get_caption_model_processor(
    model_name='florence2',
    model_name_or_path='weights/icon_caption_florence',
    device=device
)
print('✅ Florence2 모델 로드 완료')

## 4. 이미지 파싱

샘플 이미지로 테스트하거나, 직접 업로드한 스크린샷을 사용할 수 있습니다.

In [ ]:
import base64, io
import matplotlib.pyplot as plt

# --- 이미지 선택 ---
# 옵션 A: 레포에 포함된 샘플 이미지 사용
image_path = 'imgs/windows_home.png'

# 옵션 B: Colab 파일 업로드 사용 시 아래 주석 해제
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]

# --- 파싱 설정 ---
image = Image.open(image_path)
print(f'이미지 크기: {image.size}')

box_overlay_ratio = max(image.size) / 3200
draw_bbox_config = {
    'text_scale': 0.8 * box_overlay_ratio,
    'text_thickness': max(int(2 * box_overlay_ratio), 1),
    'text_padding': max(int(3 * box_overlay_ratio), 1),
    'thickness': max(int(3 * box_overlay_ratio), 1),
}
BOX_TRESHOLD = 0.05

# --- OCR ---
import time
t0 = time.time()
ocr_bbox_rslt, _ = check_ocr_box(
    image_path,
    display_img=False,
    output_bb_format='xyxy',
    goal_filtering=None,
    easyocr_args={'paragraph': False, 'text_threshold': 0.9},
    use_paddleocr=True
)
text, ocr_bbox = ocr_bbox_rslt
print(f'OCR 완료: {time.time()-t0:.1f}s')

# --- 아이콘 감지 + 캡션 ---
t1 = time.time()
labeled_img_b64, label_coords, parsed_content_list = get_som_labeled_img(
    image_path, som_model,
    BOX_TRESHOLD=BOX_TRESHOLD,
    output_coord_in_ratio=True,
    ocr_bbox=ocr_bbox,
    draw_bbox_config=draw_bbox_config,
    caption_model_processor=caption_model_processor,
    ocr_text=text,
    use_local_semantics=True,
    iou_threshold=0.7,
    scale_img=False,
    batch_size=128
)
print(f'파싱 완료: {time.time()-t1:.1f}s  |  감지된 요소: {len(parsed_content_list)}개')

## 5. 결과 시각화

In [ ]:
import base64, io
import matplotlib.pyplot as plt
from PIL import Image

result_img = Image.open(io.BytesIO(base64.b64decode(labeled_img_b64)))

plt.figure(figsize=(16, 10))
plt.imshow(result_img)
plt.axis('off')
plt.title(f'OmniParser v2 결과 — {len(parsed_content_list)}개 요소 감지', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame(parsed_content_list)
df.index.name = 'ID'
df